# DNA Methylation (450k) Preprocessing — TCGA-BRCA
Produces 8 experimental dataset variants in `statistical_filtered/`.

Notebook location: `Thesis_v3/01_Causal_feature_extraction/Methylation/`

Methylation-specific notes:
- Raw file ~3.5 GB (~485k CpG probes x 1097 samples)
- Two-pass strategy: pass 1 = chunked variance over full file; pass 2 = load top pool
- Beta values converted to M-values: log2(β / (1-β)) — better statistical properties
- Variance thresholds applied to top-15% pool (pre-filtered for memory efficiency)
- Percentile labels refer to the top-15% pool, documented explicitly

| V | Name | Selection logic |
|---|---|---|
| 1 | ultra_conservative | Variance > 98.3 pct of pool |
| 2 | conservative | Variance > 97.5 pct of pool |
| 3 | standard | Variance > 95.9 pct of pool |
| 4 | fdr_significant | Spearman FDR < 0.05, top 1000 by composite |
| 5 | balanced | Var top 25% then Spearman top 10% |
| 6 | correlation | Abs. Spearman > 97.5 pct |
| 7 | top_correlated | Top 100 positive + top 100 negative Spearman |
| 8 | composite | Average rank of Spearman + MI + Distance corr |


In [2]:
from pathlib import Path

# Notebook: Thesis_v3/01_Causal_feature_extraction/Methylation/methylation_preprocessing.ipynb
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "TCGA-BRCA.methylation450.tsv").exists()
)

METH_FILE    = BASE_DIR / "data" / "TCGA-BRCA.methylation450.tsv"
OUTCOME_FILE = BASE_DIR / "data" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "01_Causal_feature_extraction" / "Methylation" / "statistical_filtered"
CACHE_DIR    = BASE_DIR / "01_Causal_feature_extraction" / "Methylation"

VAR_CACHE    = CACHE_DIR / "meth_variance_cache.csv"       # pass 1: all probes
POOL_CACHE   = CACHE_DIR / "meth_pool_cache.csv"           # pass 2: top-15% M-values
STATS_CACHE  = CACHE_DIR / "meth_statistics_cache.csv"     # Spearman+MI+Dcor

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE   = 5000     # probes per chunk for pass 1
POOL_PCT     = 0.15     # load top 15% by variance into memory
MIN_PROBES   = 50
TARGET_BROAD = 1000

assert METH_FILE.exists(),    f"Not found: {METH_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base        : {BASE_DIR}")
print(f"Output      : {OUTPUT_DIR}")
print(f"Var cache   : {VAR_CACHE}")
print(f"Pool cache  : {POOL_CACHE}")
print(f"Stats cache : {STATS_CACHE}")
print("Paths OK")


Base        : C:\Users\olegk\Desktop\Thesis_v3
Output      : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation\statistical_filtered
Var cache   : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation\meth_variance_cache.csv
Pool cache  : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation\meth_pool_cache.csv
Stats cache : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation\meth_statistics_cache.csv
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")


In [4]:
print("Loading outcome and reading file header...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Outcome samples : {len(outcome)}")
print(f"  OS.time         : {outcome['OS.time'].min():.0f} - {outcome['OS.time'].max():.0f} days")
print(f"  Events          : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

# Read header only (fast) to get sample IDs
with open(METH_FILE) as f:
    header_samples = f.readline().strip().split("\t")[1:]

common_samples = sorted(set(header_samples) & set(outcome.index))
print(f"  File samples    : {len(header_samples)}")
print(f"  Overlap         : {len(common_samples)}")

# Count total probes (Windows-compatible)
print("  Counting probes...")
n_lines = 0
with open(METH_FILE, "rb") as f:
    for _ in f:
        n_lines += 1
n_probes_total = n_lines - 1
print(f"  Total probes    : {n_probes_total:,}")


Loading outcome and reading file header...
  Outcome samples : 1076
  OS.time         : 1 - 8605 days
  Events          : 150 (13.9%)
  File samples    : 893
  Overlap         : 772
  Counting probes...
  Total probes    : 486,427


In [5]:
import time

if VAR_CACHE.exists():
    print(f"Loading cached variance ({VAR_CACHE.name})...")
    var_cache_df = pd.read_csv(VAR_CACHE, index_col=0)
    var_series   = var_cache_df["variance"]
    miss_series  = var_cache_df["missing_frac"]
    print(f"  Loaded: {len(var_series):,} probes")
else:
    print(f"Pass 1: computing M-value variance for {n_probes_total:,} probes")
    print(f"  Chunk size: {CHUNK_SIZE}  |  Expected time: 10-20 min")
    t0 = time.time()

    sample_set = set(common_samples)
    probe_vars, probe_miss = {}, {}
    n_done = 0

    # Read header to get exact column names
    all_cols     = pd.read_csv(METH_FILE, sep="\t", nrows=0).columns.tolist()
    probe_col    = all_cols[0]   # "Composite Element REF"
    sample_cols  = [c for c in all_cols[1:] if c in sample_set]
    cols_to_read = [probe_col] + sample_cols

    reader = pd.read_csv(METH_FILE, sep="\t", index_col=0,
                         usecols=cols_to_read, chunksize=CHUNK_SIZE)
    for chunk in reader:
        chunk = chunk.astype(float)
        # Beta -> M-value conversion (clip avoids log(0))
        beta_clip = chunk.clip(1e-6, 1 - 1e-6)
        mvals = np.log2(beta_clip / (1 - beta_clip))
        for probe in mvals.index.unique():
            row = mvals.loc[probe]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]  # duplicate probe IDs — keep first row
            row_clean = row.dropna()
            probe_vars[probe]  = float(row_clean.var()) if len(row_clean) > 1 else 0.0
            chunk_row = chunk.loc[probe]
            if isinstance(chunk_row, pd.DataFrame):
                chunk_row = chunk_row.iloc[0]
            probe_miss[probe]  = float(chunk_row.isna().mean())
        n_done += len(chunk)
        if n_done % 50000 == 0:
            print(f"  {n_done:,} / {n_probes_total:,}  ({(time.time()-t0)/60:.1f} min)")

    var_series  = pd.Series(probe_vars,  name="variance")
    miss_series = pd.Series(probe_miss,  name="missing_frac")
    pd.DataFrame({"variance": var_series, "missing_frac": miss_series}).to_csv(VAR_CACHE)
    print(f"  Cached to {VAR_CACHE.name}  ({(time.time()-t0)/60:.1f} min)")

# Drop probes with >20% missing
good_mask = miss_series <= 0.20
var_good  = var_series[good_mask].sort_values(ascending=False)
print(f"  After >20% missing filter: {len(var_good):,} probes")
print(f"  Variance range: [{var_good.min():.4f}, {var_good.max():.4f}]")


Pass 1: computing M-value variance for 486,427 probes
  Chunk size: 5000  |  Expected time: 10-20 min
  50,000 / 486,427  (0.2 min)
  100,000 / 486,427  (0.3 min)
  150,000 / 486,427  (0.5 min)
  200,000 / 486,427  (0.6 min)
  250,000 / 486,427  (0.7 min)
  300,000 / 486,427  (0.9 min)
  350,000 / 486,427  (1.0 min)
  400,000 / 486,427  (1.2 min)
  450,000 / 486,427  (1.4 min)
  Cached to meth_variance_cache.csv  (1.5 min)
  After >20% missing filter: 403,060 probes
  Variance range: [0.0214, 19.2480]


In [6]:
if POOL_CACHE.exists():
    print(f"Loading pool cache ({POOL_CACHE.name})...")
    meth_top = pd.read_csv(POOL_CACHE, index_col=0)
    print(f"  Loaded: {meth_top.shape}  (samples x probes)")
else:
    top_threshold  = var_good.quantile(1 - POOL_PCT)
    probes_to_load = var_good[var_good >= top_threshold].index.tolist()
    print(f"Pass 2: loading top {POOL_PCT*100:.0f}% variance probes: {len(probes_to_load):,}")

    probes_set  = set(probes_to_load)
    sample_set  = set(common_samples)
    t0 = time.time()
    chunks_out = []
    # usecols: always keep the first (index) column + only the sample columns we need
    all_cols = pd.read_csv(METH_FILE, sep="\t", nrows=0).columns.tolist()
    cols_to_read = [all_cols[0]] + [c for c in all_cols[1:] if c in sample_set]
    reader = pd.read_csv(METH_FILE, sep="\t", index_col=0,
                         usecols=cols_to_read, chunksize=CHUNK_SIZE)
    for chunk in reader:
        keep = chunk.index.intersection(probes_set)
        if len(keep):
            chunks_out.append(chunk.loc[keep])
    print(f"  Chunks collected: {len(chunks_out)}")

    meth_raw  = pd.concat(chunks_out)
    cols_keep = [c for c in common_samples if c in meth_raw.columns]
    beta_clip = meth_raw[cols_keep].astype(float).clip(1e-6, 1 - 1e-6)
    mvals     = np.log2(beta_clip / (1 - beta_clip))
    meth_top  = mvals.T   # samples x probes

    imp      = SimpleImputer(strategy="median")
    meth_top = pd.DataFrame(imp.fit_transform(meth_top),
                            index=meth_top.index, columns=meth_top.columns)
    meth_top.to_csv(POOL_CACHE)
    print(f"  Cached to {POOL_CACHE.name}  ({(time.time()-t0)/60:.1f} min)")

common2 = meth_top.index.intersection(outcome.index)
meth_top = meth_top.loc[common2]
outcome  = outcome.loc[common2]
print(f"  Final shape : {meth_top.shape}  (samples x probes)")
print(f"  Aligned outcome: {outcome.shape}")


Pass 2: loading top 15% variance probes: 60,459
  Chunks collected: 97
  Cached to meth_pool_cache.csv  (1.6 min)
  Final shape : (772, 60459)  (samples x probes)
  Aligned outcome: (772, 2)


In [7]:
if STATS_CACHE.exists():
    print(f"Loading cached statistics ({STATS_CACHE.name})...")
    stats = pd.read_csv(STATS_CACHE)
    print(f"  Loaded: {len(stats):,} probes")
else:
    import time
    os_time = outcome["OS.time"].values
    probes  = meth_top.columns.tolist()
    n       = len(probes)
    print(f"Computing statistics for {n:,} probes...")

    # 1. Spearman correlation
    print("  [1/3] Spearman correlation...")
    t0 = time.time()
    rho_arr, pval_arr = [], []
    for i, p in enumerate(probes):
        r, pv = spearmanr(meth_top[p].values, os_time)
        rho_arr.append(0.0 if np.isnan(r) else float(r))
        pval_arr.append(1.0 if np.isnan(pv) else float(pv))
        if (i + 1) % 5000 == 0:
            print(f"    {i+1:,} / {n:,}  ({time.time()-t0:.0f}s)")
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")

    # 2. Mutual information
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(
        meth_top.values, os_time, random_state=42, n_neighbors=5
    )

    # 3. Distance correlation (rank-based approximation)
    # Consistent with RNA, CNV, Mutation pipelines.
    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = []
    for p in probes:
        x_rank = rankdata(meth_top[p].values).astype(float)
        dc_arr.append(abs(float(np.corrcoef(x_rank, os_rank)[0, 1])))

    stats = pd.DataFrame({
        "probe":         probes,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "variance":      var_good.reindex(meth_top.columns).values,
    })

    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (
        stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]
    ) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE.name}")

n_fdr = int((stats["fdr"] < 0.05).sum())
print(f"  FDR < 0.05  : {n_fdr:,} / {len(stats):,} probes")
print(f"  Top 10 by composite score:")
print(stats[["probe","abs_spearman","mutual_info","distance_corr","composite"]].head(10).to_string(index=False))


Computing statistics for 60,459 probes...
  [1/3] Spearman correlation...
    5,000 / 60,459  (4s)
    10,000 / 60,459  (7s)
    15,000 / 60,459  (10s)
    20,000 / 60,459  (14s)
    25,000 / 60,459  (17s)
    30,000 / 60,459  (21s)
    35,000 / 60,459  (24s)
    40,000 / 60,459  (28s)
    45,000 / 60,459  (31s)
    50,000 / 60,459  (35s)
    55,000 / 60,459  (38s)
    60,000 / 60,459  (42s)
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to meth_statistics_cache.csv
  FDR < 0.05  : 26 / 60,459 probes
  Top 10 by composite score:
     probe  abs_spearman  mutual_info  distance_corr  composite
cg05619116      0.161697     0.071054       0.161697  18.333333
cg17178175      0.133453     0.056453       0.133453 146.666667
cg00101629      0.132922     0.056074       0.132922 157.000000
cg23692472      0.138584     0.053660       0.138584 160.000000
cg26625629      0.167857     0.051606       0.167857 162.000000
cg03612333      0.135294     0.05239

In [8]:
print("Creating 8 dataset variants...")
print(f"Output: {OUTPUT_DIR}")
print(f"Note: variance thresholds applied to top-{POOL_PCT*100:.0f}% pool (~{len(stats):,} probes)")

def build(probe_list, min_p=MIN_PROBES):
    """Select probes; pad with top composite probes if below minimum.
    No normalization — M-values are already on a symmetric log scale."""
    probe_list = [p for p in probe_list if p in meth_top.columns]
    if len(probe_list) < min_p:
        have = set(probe_list)
        probe_list += [p for p in stats["probe"] if p not in have][:min_p - len(probe_list)]
    return meth_top[probe_list]

created = []

# V1 Ultra-Conservative: variance > 98.3 pct of pool
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.983)]["probe"].tolist())
fname = f"meth_1_ultra_conservative_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V1","name":"ultra_conservative","n":len(df.columns),"logic":"Variance > 98.3 pct (pool)","file":fname})
print(f"  V1 ultra_conservative : {len(df.columns):,} probes")

# V2 Conservative: variance > 97.5 pct of pool
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.975)]["probe"].tolist())
fname = f"meth_2_conservative_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V2","name":"conservative","n":len(df.columns),"logic":"Variance > 97.5 pct (pool)","file":fname})
print(f"  V2 conservative       : {len(df.columns):,} probes")

# V3 Standard: variance > 95.9 pct of pool
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.959)]["probe"].tolist())
fname = f"meth_3_standard_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V3","name":"standard","n":len(df.columns),"logic":"Variance > 95.9 pct (pool)","file":fname})
print(f"  V3 standard           : {len(df.columns):,} probes")

# V4 FDR-significant: top TARGET_BROAD by composite (FDR fallback if 0)
fdr_probes = stats[stats["fdr"] < 0.05].head(TARGET_BROAD)["probe"].tolist()
df    = build(fdr_probes, min_p=TARGET_BROAD)
fname = f"meth_4_fdr_significant_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V4","name":"fdr_significant","n":len(df.columns),"logic":"FDR<0.05, top 1000 composite fallback","file":fname})
print(f"  V4 fdr_significant    : {len(df.columns):,} probes  (raw FDR<0.05: {int((stats['fdr']<0.05).sum())})")

# V5 Balanced: var top 25% then Spearman top 10%
var_sub     = stats[stats["variance"] > stats["variance"].quantile(0.75)]
corr_thresh = var_sub["abs_spearman"].quantile(0.90)
df    = build(var_sub[var_sub["abs_spearman"] > corr_thresh]["probe"].tolist())
fname = f"meth_5_balanced_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V5","name":"balanced","n":len(df.columns),"logic":"Var top 25% -> Spearman top 10%","file":fname})
print(f"  V5 balanced           : {len(df.columns):,} probes")

# V6 Correlation-based: abs Spearman > 97.5 pct
df    = build(stats[stats["abs_spearman"] > stats["abs_spearman"].quantile(0.975)]["probe"].tolist())
fname = f"meth_6_correlation_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V6","name":"correlation","n":len(df.columns),"logic":"Abs Spearman > 97.5 pct","file":fname})
print(f"  V6 correlation        : {len(df.columns):,} probes")

# V7 Top Correlated: top 100 pos + top 100 neg Spearman
top_pos = stats[stats["spearman_rho"] > 0].head(100)["probe"].tolist()
top_neg = stats[stats["spearman_rho"] < 0].tail(100)["probe"].tolist()
df      = build(sorted(set(top_pos + top_neg)))
stats[stats["probe"].isin(set(top_pos + top_neg))][
    ["probe","spearman_rho","pval","fdr","variance"]
].to_csv(OUTPUT_DIR / "meth_7_top_correlated_annotated.csv", index=False)
fname = f"meth_7_top_correlated_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V7","name":"top_correlated","n":len(df.columns),"logic":"Top 100 pos + top 100 neg Spearman","file":fname})
print(f"  V7 top_correlated     : {len(df.columns):,} probes")

# V8 Composite: top TARGET_BROAD by avg rank Spearman+MI+Dcor
df    = build(stats.head(TARGET_BROAD)["probe"].tolist(), min_p=TARGET_BROAD)
fname = f"meth_8_composite_{len(df.columns)}probes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V8","name":"composite","n":len(df.columns),"logic":"Avg rank Spearman+MI+Dcor, top 1000","file":fname})
print(f"  V8 composite          : {len(df.columns):,} probes")

# Auxiliary files
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "meth_statistics_complete.csv", index=False)
summary = pd.DataFrame(created)
summary.to_csv(OUTPUT_DIR / "datasets_summary.csv", index=False)

print()
print(summary[["V","name","n","logic"]].to_string(index=False))
print(f"Saved to: {OUTPUT_DIR}")


Creating 8 dataset variants...
Output: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation\statistical_filtered
Note: variance thresholds applied to top-15% pool (~60,459 probes)
  V1 ultra_conservative : 1,028 probes
  V2 conservative       : 1,512 probes
  V3 standard           : 2,479 probes
  V4 fdr_significant    : 1,000 probes  (raw FDR<0.05: 26)
  V5 balanced           : 1,512 probes
  V6 correlation        : 1,512 probes
  V7 top_correlated     : 200 probes
  V8 composite          : 1,000 probes

 V               name    n                                 logic
V1 ultra_conservative 1028            Variance > 98.3 pct (pool)
V2       conservative 1512            Variance > 97.5 pct (pool)
V3           standard 2479            Variance > 95.9 pct (pool)
V4    fdr_significant 1000 FDR<0.05, top 1000 composite fallback
V5           balanced 1512       Var top 25% -> Spearman top 10%
V6        correlation 1512               Abs Spearman > 97.5 pct
V7     top_co